# A full training

# Install

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [132]:
# !pip install datasets evaluate transformers[sentencepiece]
# !pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl

# Check GPU is connected

In [133]:
import torch

In [134]:
!nvidia-smi

Mon Jul  1 13:43:05 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 552.23                 Driver Version: 552.23         CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA T1000 8GB             WDDM  |   00000000:65:00.0 Off |                  N/A |
| 33%   35C    P8             N/A /   50W |    5436MiB /   8192MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [135]:

if torch.cuda.is_available():
    print("CUDA is available! 🎉")
    print("Number of CUDA devices:", torch.cuda.device_count())
else:
    print("CUDA is not available. 🙁")

CUDA is available! 🎉
Number of CUDA devices: 1


In [136]:
import torch
import sys
print('__Python VERSION:', sys.version)
print('__pyTorch VERSION:', torch.__version__)
print('__CUDA VERSION')
from subprocess import call
# call(["nvcc", "--version"]) does not work
! nvcc --version
print('__CUDNN VERSION:', torch.backends.cudnn.version())
print('__Number CUDA Devices:', torch.cuda.device_count())
print('__Devices')
call(["nvidia-smi", "--format=csv", "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free"])
print('Active CUDA Device: GPU', torch.cuda.current_device())
print ('Available devices ', torch.cuda.device_count())
print ('Current cuda device ', torch.cuda.current_device())

__Python VERSION: 3.11.9 | packaged by Anaconda, Inc. | (main, Apr 19 2024, 16:40:41) [MSC v.1916 64 bit (AMD64)]
__pyTorch VERSION: 2.5.0.dev20240626+cu124
__CUDA VERSION
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2021 NVIDIA Corporation
Built on Fri_Dec_17_18:28:54_Pacific_Standard_Time_2021
Cuda compilation tools, release 11.6, V11.6.55
Build cuda_11.6.r11.6/compiler.30794723_0
__CUDNN VERSION: 90100
__Number CUDA Devices: 1
__Devices
Active CUDA Device: GPU 0
Available devices  1
Current cuda device  0


In [137]:
# activate GPU device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


cuda:0


# Load dataset

In [138]:
from pathlib import Path
# Get the current working directory
cwd = Path.cwd()

# Get the path to the parent directory
parent_dir = cwd.parent

# Add the parent directory to the Python path
sys.path.append(str(parent_dir))

In [139]:
parent_dir = Path(cwd).parent

folder_path = str(parent_dir / 'fine_tuning_dataset_repository\dataset\preprocess_SYNCH_grouping_by_definition_0627.csv')


In [140]:
from datasets import load_dataset
from datasets import Dataset 
from datasets import DatasetDict
import pandas as pd

my_dataset = pd.read_csv(folder_path)
# my_dataset.head(5)

my_dataset = Dataset.from_pandas(my_dataset)

# print(raw_datasets)
# print(raw_datasets[0])

train_test_dataset = my_dataset.train_test_split(test_size=0.9)
test_valid = train_test_dataset['test'].train_test_split(test_size=0.9)

# raw_train_dataset = raw_datasets["train"]

# gather everyone if you want to have a single DatasetDict
raw_datasets = DatasetDict({
    'train': train_test_dataset['train'],
    'test': test_valid['test'],
    'validation': test_valid['train']})


In [141]:
from datasets import Dataset
# raw_datasets = load_dataset("glue","mrpc")
# raw_datasets = load_dataset("humarin/chatgpt-paraphrases")



In [142]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 13
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1
    })
})

In [143]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'desire to stop family cycle of violence and alcohol misuse',
 'sentence2': 'wanting the violence by a partner or in a family to stop',
 'label': 1,
 'idx': 8}

In [144]:
raw_train_dataset.features

{'sentence1': Value(dtype='string', id=None),
 'sentence2': Value(dtype='string', id=None),
 'label': Value(dtype='int64', id=None),
 'idx': Value(dtype='int64', id=None)}

# Use a pretrained model

In [145]:
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModel
import torch
import torch.nn.functional as F

In [146]:

checkpoint = 'sentence-transformers/all-mpnet-base-v2'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

model.to(device) # Move model to the device

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [147]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map: 100%|██████████| 1/1 [00:00<00:00, 116.89 examples/s]


# prepare for training

In [148]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets['train'].column_names

['labels', 'input_ids', 'attention_mask']

In [149]:
# type(tokenized_datasets)

In [150]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
[len(x) for x in samples["input_ids"]]

[27]

In [151]:
# ["attention_mask", "input_ids", "labels", "token_type_ids"]

In [152]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

In [153]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([1]),
 'input_ids': torch.Size([1, 27]),
 'attention_mask': torch.Size([1, 27])}

In [154]:
# Move the batch of data to the same device
for key in batch.keys():
    batch[key] = batch[key].to(device)

# evaluate

In [155]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")


In [156]:
from transformers import AutoModelForSequenceClassification

# model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

In [157]:
from datasets import load_metric

metric = load_metric("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)



ValueError: The repository for accuracy contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/accuracy.
Please pass the argument `trust_remote_code=True` to allow custom code to be run.

In [158]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    # train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [128]:
trainer.train()

ValueError: Trainer: training requires a train_dataset.

In [130]:
trainer.evaluate()

100%|██████████| 1/1 [00:00<00:00, 1002.70it/s]


{'eval_runtime': 0.1406,
 'eval_samples_per_second': 7.111,
 'eval_steps_per_second': 7.111}

In [89]:
predictions = model.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

AttributeError: 'MPNetModel' object has no attribute 'predict'

In [ ]:
predictions.predictions

In [ ]:
validation_dataset = tokenized_datasets["validation"]
validation_dataset[0]

In [ ]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)
preds

In [ ]:
raw_datasets

In [ ]:
raw_validation_dataset = raw_datasets["validation"]
raw_validation_dataset[0]



In [ ]:
predictions.predictions

# evaluate

In [124]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'logits'

## Using trainer 

In [ ]:
trainer.evaluate()

100%|██████████| 1/1 [00:00<00:00, 1002.94it/s]


{'eval_runtime': 0.3286,
 'eval_samples_per_second': 3.043,
 'eval_steps_per_second': 3.043}

# Inference

Let's test the model on a new sentence:

In [ ]:
raw_datasets['validation'][0]

{'sentence1': 'community intimate partner violence rate ',
 'sentence2': 'family and friend permissive intimate partner violence norms',
 'label': 1,
 'idx': 2}

In [ ]:
sentence1 = "child welfare system intervention"
sentence2 = "children removed from house"
sentence3  = "I am climbing and looking for a climbing partner"

In [ ]:

encoding = tokenizer(sentence1, sentence2, return_tensors="pt")
encoding = {k: v.to(trainer.model.device) for k,v in encoding.items()}

outputs = trainer.model(**encoding)
print(outputs)



BaseModelOutputWithPooling(last_hidden_state=tensor([[[ 0.0323, -0.0830, -0.0017,  ...,  0.1646, -0.0992,  0.1524],
         [-0.0683, -0.0477, -0.0067,  ...,  0.1092, -0.1719,  0.2046],
         [ 0.0177,  0.0823,  0.0718,  ...,  0.1510, -0.1021,  0.2635],
         ...,
         [ 0.0773, -0.0699,  0.1049,  ...,  0.2227, -0.1274,  0.2081],
         [ 0.1323,  0.0447,  0.0788,  ...,  0.2172, -0.1440,  0.1304],
         [ 0.1499, -0.0492,  0.0353,  ...,  0.1842, -0.1158,  0.1559]]],
       device='cuda:0', grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[-9.8332e-03,  9.9304e-04, -8.8013e-02, -8.5168e-03,  2.7329e-02,
         -3.4403e-02, -1.9863e-02, -5.7007e-02,  6.5649e-02,  4.0545e-02,
         -4.3961e-02,  1.4975e-03,  4.4875e-02,  9.4936e-02,  3.8503e-02,
          5.7530e-03, -1.8841e-02,  1.0910e-01,  8.1893e-02,  1.1882e-01,
         -2.8460e-02,  4.0943e-02, -4.9797e-02,  3.4968e-02,  1.4050e-02,
         -1.4423e-02, -1.0911e-05,  1.8997e-02, -4.6818e-02,  6.6692

The logits that come out of the model are of shape (batch_size, num_labels). As we are only forwarding a single sentence through the model, the `batch_size` equals 1. The logits is a tensor that contains the (unnormalized) scores for every individual label.

In [ ]:
logits = outputs.logits
print(logits)
predictions = torch.argmax(logits, dim=-1)

# logits.shape
print(predictions)

AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'logits'

To turn them into actual predicted labels, we first apply a sigmoid function independently to every score, such that every score is turned into a number between 0 and 1, that can be interpreted as a "probability" for how certain the model is that a given class belongs to the input text.

Next, we use a threshold (typically, 0.5) to turn every probability into either a 1 (which means, we predict the label for the given example) or a 0 (which means, we don't predict the label for the given example).

In [ ]:
# # apply sigmoid + threshold
# sigmoid = torch.nn.Sigmoid()
# probs = sigmoid(logits.squeeze().cpu())
# predictions = np.zeros(probs.shape)
# predictions[np.where(probs >= 0.5)] = 1
# # turn predicted id's into actual label names
# predicted_labels = [id2label[idx] for idx, label in enumerate(predictions) if label == 1.0]
# print(predicted_labels)

In [ ]:
trainer.save_model("./my_model")

In [ ]:
model.save_to_hub("my_new_model")

# Use the pretrained model for PHRAME

In [ ]:
sentence1 = "Im happy to try to train a model that works for semantic similarity"
sentence2 = "trainig a model that workds for semantic similarity makes me happy"
sentence3  = "I am climbing and looking for a climbing partner"


# load pretrained model

In [ ]:
model = AutoModel.from_pretrained("./my_model")

In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers import SentenceTransformer, models, InputExample, losses


In [ ]:
model = SentenceTransformer("./my_model")


In [ ]:
from huggingface_hub import notebook_login

In [ ]:
notebook_login("./my_model")

In [ ]:
trainer.push_to_hub("your-username/model-name")

# (IGNORE) training without trainer

In [ ]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

In [ ]:
from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

In [ ]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)